In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import sys
import os

PROJECT_ROOT = "/content/drive/MyDrive/Semiconductor-AnomalyDetection"

sys.path.append(PROJECT_ROOT)

좋은 feature들만 남은 데이터셋으로 모델 간 비교 (logistic vs random_forest vs xgboost)

In [5]:
import pandas as pd

processed_path = os.path.join(PROJECT_ROOT, "data", "processed")

X_train = pd.read_parquet(os.path.join(processed_path, "X_train_selected.parquet"))
X_test = pd.read_parquet(os.path.join(processed_path, "X_test_selected.parquet"))

y_train = pd.read_parquet(os.path.join(processed_path, "y_train.parquet"))["label"]
y_test = pd.read_parquet(os.path.join(processed_path, "y_test.parquet"))["label"]

print(X_train.shape)
print(X_test.shape)
print(X_train.head())
print(y_train.head())

(1253, 50)
(314, 50)
      sensor_015  sensor_022  sensor_023  sensor_029  sensor_060  sensor_064  \
1198      3.0143    -5631.75     2667.50     77.3444     -1.8600     17.9545   
436      10.2077    -5474.25     2997.00     59.9778     -8.3955     21.3864   
635      10.1995    -5438.50     2701.25     67.4778      2.1145     16.7345   
996       8.4954    -5525.00     2691.50     69.9444     -3.0882     14.7591   
782      12.3992    -5544.25     2771.50     76.3556     -1.1427     13.1664   

      sensor_065  sensor_066  sensor_071  sensor_096  ...  sensor_435  \
1198     21.8600     27.0072    622.6591      0.0001  ...     15.9843   
436      28.3955     35.1195    596.9536      0.0002  ...     14.5020   
635      17.8855     26.4123    621.0364      0.0001  ...      6.4037   
996      23.0882     30.6344    614.7945      0.0000  ...     11.1987   
782      21.1427     29.0083    622.9400      0.0000  ...     22.3190   

      sensor_436  sensor_437  sensor_453  sensor_456  senso

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from src.evaluation.metrics import evaluate_binary

In [17]:

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:, 1]

print(log_prob)
log_metrics = evaluate_binary(y_test, log_pred)
print(log_metrics)


inf train: 0
inf test: 0
y_train NaN: 0
y_test NaN: 0
X_train shape: (1253, 50)
y_train shape: (1253,)
y_train labels: label
-1    1170
 1      83
Name: count, dtype: int64


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]